# Notebook 9: Regression Metrics

**Difficulty:** ⭐⭐⭐ | **Estimated Time:** 2.5 hours | **Week 7 — Model Evaluation, Validation & Metrics**

---

## Why This Matters

Imagine you build a model that predicts **how much money will be spent in a credit card transaction**. The model makes predictions. But how do you know if those predictions are *good*? Is being off by $5 acceptable? What about $500? What about a single transaction where you're off by $50,000?

Different regression metrics answer these questions differently. Using the wrong metric can give you a dangerously false sense of confidence — or make a great model look mediocre.

---

## Real-World Analogy: Different Rulers Measure Different Things

Think of measuring the gap between your predictions and reality like choosing a ruler:

- **MAE (Mean Absolute Error):** A regular ruler. Measures the average distance from predicted to actual. Simple, honest, treats all gaps equally.
- **RMSE (Root Mean Squared Error):** A ruler that *stretches* when it encounters large gaps. A 10-cm error gets recorded as 100 cm² before taking the root — big mistakes are punished extra hard.
- **R²:** Not a ruler at all, but a score card: "what fraction of the variation in reality does your model actually explain?"
- **Adjusted R²:** The same scorecard but it deducts points when you add useless rulers to your toolkit.
- **MAPE:** A ruler that measures in *percentages* — useful when you need to communicate with stakeholders who care about proportional errors, not absolute ones.

---

## Theme: Predicting Credit Card Transaction Amounts

Our scenario: A fraud detection team wants to predict the **dollar amount** of a transaction before fully processing it. Unusually large transactions trigger extra verification steps. We'll train regression models and evaluate them using every major regression metric.

## Section 1: Setup and Data Generation

We generate 1,000 synthetic credit card transactions. Transaction amounts in the real world follow a **log-normal distribution** — most transactions are small (coffee, groceries), with a long tail of large ones (electronics, travel).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
np.random.seed(42)

# Style settings for all plots
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11
sns.set_theme(style='whitegrid', palette='muted')

print("Libraries loaded successfully.")

In [ ]:
# ---------------------------------------------------------------
# Generate synthetic transaction data
# ---------------------------------------------------------------
n_samples = 1000

# Features: merchant category code, hour of day, days since last transaction
merchant_category = np.random.randint(1, 20, n_samples).astype(float)   # 1-19 MCC groups
hour_of_day       = np.random.randint(0, 24, n_samples).astype(float)    # 0-23
days_since_last   = np.random.exponential(scale=5, size=n_samples)       # avg 5 days

# True transaction amount: log-normal with feature-driven mean
# Higher MCC and evening hours increase amount slightly
log_mean = (
    3.5
    + 0.05 * merchant_category
    + 0.02 * np.abs(hour_of_day - 12)   # peak at midday and midnight
    + 0.01 * days_since_last
)
y_true = np.random.lognormal(mean=log_mean, sigma=0.8)

# Build feature matrix X
X = np.column_stack([merchant_category, hour_of_day, days_since_last])
feature_names = ['merchant_category', 'hour_of_day', 'days_since_last']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_true, test_size=0.2, random_state=42
)

print(f"Dataset shape: {X.shape}")
print(f"Transaction amount — min: ${y_true.min():.2f}  max: ${y_true.max():.2f}  median: ${np.median(y_true):.2f}")
print(f"Train size: {X_train.shape[0]} | Test size: {X_test.shape[0]}")

In [ ]:
# Visualise the distribution of transaction amounts
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Raw amounts — log-normal has a heavy right tail
axes[0].hist(y_true, bins=60, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].set_title('Transaction Amount Distribution (Raw)', fontweight='bold')
axes[0].set_xlabel('Transaction Amount ($)')
axes[0].set_ylabel('Count')
axes[0].axvline(np.median(y_true), color='tomato', linestyle='--',
                label=f'Median = ${np.median(y_true):.0f}')
axes[0].axvline(np.mean(y_true), color='orange', linestyle='--',
                label=f'Mean = ${np.mean(y_true):.0f}')
axes[0].legend()

# Log scale makes the distribution easier to see
axes[1].hist(np.log(y_true + 1), bins=50, color='mediumseagreen', edgecolor='white', alpha=0.85)
axes[1].set_title('log(Transaction Amount) — Near-Normal', fontweight='bold')
axes[1].set_xlabel('log(Amount + 1)')
axes[1].set_ylabel('Count')

plt.suptitle('Credit Card Transaction Amounts (1,000 Synthetic Transactions)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print("\nNote: log-normal distributions are typical for financial transaction data.")

## Section 2: Train Two Models

We train a simple **Linear Regression** (ok fit) and a **Polynomial Regression** (degree 2, better fit). We'll compare metrics across both.

In [ ]:
# ---------------------------------------------------------------
# Model 1: Plain Linear Regression
# ---------------------------------------------------------------
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)

# ---------------------------------------------------------------
# Model 2: Polynomial Regression (degree=2) via Pipeline
# ---------------------------------------------------------------
poly_model = Pipeline([
    ('poly_features', PolynomialFeatures(degree=2, include_bias=False)),
    ('linear_reg',    LinearRegression())
])
poly_model.fit(X_train, y_train)
y_pred_poly = poly_model.predict(X_test)

# Clip negative predictions (amounts can't be negative)
y_pred_lr   = np.clip(y_pred_lr,   0, None)
y_pred_poly = np.clip(y_pred_poly, 0, None)

print("Model 1 (Linear Regression) — sample predictions vs actuals:")
comparison = pd.DataFrame({
    'Actual ($)':        y_test[:8].round(2),
    'Pred Linear ($)':  y_pred_lr[:8].round(2),
    'Pred Poly ($)':    y_pred_poly[:8].round(2),
})
print(comparison.to_string(index=False))

## Section 3: Implementing All Metrics from Scratch

### 3.1 MAE — Mean Absolute Error

$$\text{MAE} = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|$$

**Plain English:** On average, how many *dollars* off are our predictions?

**Key properties:**
- Same units as the target variable (dollars in our case) — very interpretable.
- **Robust to outliers:** a \$10,000 error counts as one large error, not 10,000².
- Every error contributes equally regardless of size.

**When to use:** When outliers shouldn't dominate your evaluation and interpretability matters.

In [ ]:
# ---------------------------------------------------------------
# Implement all metrics from scratch, then verify vs sklearn
# ---------------------------------------------------------------

def mae_scratch(y_true, y_pred):
    """Mean Absolute Error: average of absolute residuals."""
    return np.mean(np.abs(y_true - y_pred))

def mse_scratch(y_true, y_pred):
    """Mean Squared Error: average of squared residuals."""
    return np.mean((y_true - y_pred) ** 2)

def rmse_scratch(y_true, y_pred):
    """Root Mean Squared Error: square root of MSE."""
    return np.sqrt(mse_scratch(y_true, y_pred))

def r2_scratch(y_true, y_pred):
    """Coefficient of Determination: 1 - SS_res / SS_tot."""
    ss_res = np.sum((y_true - y_pred) ** 2)      # residual sum of squares
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)  # total sum of squares
    return 1 - ss_res / ss_tot

def adjusted_r2_scratch(y_true, y_pred, n_features):
    """Adjusted R²: penalises for number of features.
       adj_R² = 1 - (1-R²) * (n-1) / (n-p-1)
    """
    n = len(y_true)
    r2 = r2_scratch(y_true, y_pred)
    return 1 - (1 - r2) * (n - 1) / (n - n_features - 1)

def mape_scratch(y_true, y_pred, epsilon=1e-8):
    """Mean Absolute Percentage Error.
       epsilon avoids division by zero (handles near-zero actuals).
    """
    return np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + epsilon))) * 100

print("All metric functions defined.")

In [ ]:
# ---------------------------------------------------------------
# Compute all metrics for both models and verify vs sklearn
# ---------------------------------------------------------------
models = {
    'Linear Regression':     (y_pred_lr,   3),   # 3 original features
    'Polynomial (degree=2)': (y_pred_poly, 9),   # 3 + 3 cross + 3 squared = 9
}

results = {}
for name, (y_pred, n_feat) in models.items():
    results[name] = {
        'MAE ($)':       mae_scratch(y_test, y_pred),
        'RMSE ($)':      rmse_scratch(y_test, y_pred),
        'R²':            r2_scratch(y_test, y_pred),
        'Adjusted R²':   adjusted_r2_scratch(y_test, y_pred, n_feat),
        'MAPE (%)':      mape_scratch(y_test, y_pred),
    }

metrics_df = pd.DataFrame(results).T.round(4)
print("=== Metric Comparison: Linear vs Polynomial ===")
print(metrics_df.to_string())

# Verify MAE and RMSE vs sklearn
print("\n--- Verification vs sklearn ---")
print(f"  MAE  (scratch vs sklearn): {mae_scratch(y_test, y_pred_lr):.4f}  |  {mean_absolute_error(y_test, y_pred_lr):.4f}")
print(f"  RMSE (scratch vs sklearn): {rmse_scratch(y_test, y_pred_lr):.4f}  |  {np.sqrt(mean_squared_error(y_test, y_pred_lr)):.4f}")
print(f"  R²   (scratch vs sklearn): {r2_scratch(y_test, y_pred_lr):.4f}  |  {r2_score(y_test, y_pred_lr):.4f}")

## Section 4: MAE vs RMSE — The Outlier Effect

### 3.2 RMSE — Root Mean Squared Error

$$\text{RMSE} = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2}$$

**Plain English:** Typical error, but with large errors amplified. Because we *square* before averaging, a $\$1{,}000$ error contributes $1{,}000{,}000$ to the sum — it hurts far more than ten $\$100$ errors combined.

**Key property:** RMSE ≥ MAE always. They are equal only when *all* errors are exactly the same size.

**Demonstration:** We inject 5 extreme fraud transactions where the true amount is \$50,000 but the model predicts \$100. Watch what happens to each metric.

In [ ]:
# ---------------------------------------------------------------
# Outlier injection experiment
# Inject 5 extreme errors: true=$50,000, predicted=$100
# ---------------------------------------------------------------
y_test_outlier  = y_test.copy()
y_pred_outlier  = y_pred_lr.copy()

# Pick 5 random positions in the test set to inject the outlier error
outlier_indices = np.random.choice(len(y_test), size=5, replace=False)
y_test_outlier[outlier_indices]  = 50_000.0   # actual: huge fraud transaction
y_pred_outlier[outlier_indices]  =    100.0   # predicted: model missed it completely

# Compare metrics before and after outlier injection
outlier_comparison = pd.DataFrame({
    'Before Outliers': {
        'MAE ($)':  mae_scratch(y_test, y_pred_lr),
        'RMSE ($)': rmse_scratch(y_test, y_pred_lr),
    },
    'After 5 Outlier Errors': {
        'MAE ($)':  mae_scratch(y_test_outlier, y_pred_outlier),
        'RMSE ($)': rmse_scratch(y_test_outlier, y_pred_outlier),
    }
}).round(2)

print("=== Effect of 5 Extreme Outlier Errors ===")
print(outlier_comparison.to_string())
print()

mae_increase  = mae_scratch(y_test_outlier, y_pred_outlier)  / mae_scratch(y_test, y_pred_lr)
rmse_increase = rmse_scratch(y_test_outlier, y_pred_outlier) / rmse_scratch(y_test, y_pred_lr)
print(f"MAE  multiplied by: {mae_increase:.1f}x")
print(f"RMSE multiplied by: {rmse_increase:.1f}x  <-- RMSE is far more sensitive to outliers")

In [ ]:
# ---------------------------------------------------------------
# Visual: How MAE and RMSE respond as outlier magnitude grows
# ---------------------------------------------------------------
outlier_magnitudes = np.linspace(100, 100_000, 200)  # outlier error from $100 to $100K
mae_values, rmse_values = [], []

base_test = y_test.copy()
base_pred = y_pred_lr.copy()

for magnitude in outlier_magnitudes:
    y_t = base_test.copy()
    y_p = base_pred.copy()
    y_t[outlier_indices] = magnitude   # escalate the true outlier value
    y_p[outlier_indices] = 100.0       # model always predicts $100
    mae_values.append(mae_scratch(y_t, y_p))
    rmse_values.append(rmse_scratch(y_t, y_p))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(outlier_magnitudes / 1000, mae_values,  label='MAE',  color='steelblue', linewidth=2)
ax.plot(outlier_magnitudes / 1000, rmse_values, label='RMSE', color='tomato',    linewidth=2)
ax.set_xlabel('Outlier Error Magnitude ($000s)', fontsize=12)
ax.set_ylabel('Metric Value ($)', fontsize=12)
ax.set_title('MAE vs RMSE as Outlier Errors Grow\n(5 fraud transactions predicted as $100)', fontweight='bold')
ax.legend(fontsize=12)
ax.fill_between(outlier_magnitudes / 1000, mae_values, rmse_values,
                alpha=0.15, color='orange', label='RMSE - MAE gap')
plt.tight_layout()
plt.show()
print("Key insight: RMSE grows much faster than MAE as outlier errors increase.")

## Section 5: R² and Adjusted R²

### 5.1 R² — Coefficient of Determination

$$R^2 = 1 - \frac{SS_{\text{res}}}{SS_{\text{tot}}} = 1 - \frac{\sum(y_i - \hat{y}_i)^2}{\sum(y_i - \bar{y})^2}$$

| R² value | Interpretation |
|-----------|----------------|
| 1.0       | Perfect predictions |
| 0.85      | Model explains 85% of variance in transaction amounts |
| 0.0       | Model is no better than predicting the mean every time |
| < 0       | Model is *worse* than just predicting the mean |

### 5.2 Adjusted R² — Penalising Feature Count

$$\text{Adj.}\, R^2 = 1 - \frac{(1-R^2)(n-1)}{n-p-1}$$

where $p$ = number of features. R² **always increases** when you add features, even random noise. Adjusted R² drops when a feature adds less than it costs.

**Rule:** When comparing models with different numbers of features, always use Adjusted R².

In [ ]:
# ---------------------------------------------------------------
# Demonstrate: R² always increases; Adjusted R² eventually drops
# Add one random noise feature at a time
# ---------------------------------------------------------------
n_noise_features = 30   # we'll add up to 30 random features
r2_values, adj_r2_values, feature_counts = [], [], []

# Start with the 3 real features
X_growing = X_train.copy()
X_test_growing = X_test.copy()

for i in range(n_noise_features + 1):
    current_p = X_growing.shape[1]   # number of features at this step

    model_i = LinearRegression().fit(X_growing, y_train)
    y_pred_i = model_i.predict(X_test_growing)

    r2   = r2_scratch(y_test, y_pred_i)
    ar2  = adjusted_r2_scratch(y_test, y_pred_i, current_p)

    r2_values.append(r2)
    adj_r2_values.append(ar2)
    feature_counts.append(current_p)

    # Add one random noise feature for the next iteration
    if i < n_noise_features:
        noise_train = np.random.randn(X_growing.shape[0], 1)
        noise_test  = np.random.randn(X_test_growing.shape[0], 1)
        X_growing       = np.hstack([X_growing, noise_train])
        X_test_growing  = np.hstack([X_test_growing, noise_test])

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(feature_counts, r2_values,     label='R²',          color='steelblue', linewidth=2, marker='o', markersize=4)
ax.plot(feature_counts, adj_r2_values, label='Adjusted R²', color='tomato',    linewidth=2, marker='s', markersize=4)
ax.axvline(3, color='gray', linestyle=':', label='Original 3 features')
ax.set_xlabel('Number of Features (3 real + noise)', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('R² vs Adjusted R² as Random Features Are Added\n(R² always grows; Adjusted R² exposes the useless additions)', fontweight='bold')
ax.legend(fontsize=11)
ax.set_ylim(-0.1, 1.05)
plt.tight_layout()
plt.show()
print("Adjusted R² penalises for adding noise features — it tracks model quality, not feature quantity.")

## Section 6: MAPE — Mean Absolute Percentage Error

$$\text{MAPE} = \frac{100}{n} \sum_{i=1}^{n} \frac{|y_i - \hat{y}_i|}{|y_i|}$$

**Plain English:** On average, what *percentage* of the true amount were we off by?

**Strengths:** Scale-free — you can compare across datasets with different currencies or units. Easy to communicate: "our model is off by 12% on average."

**Fatal flaw:** Undefined when $y_i = 0$. Also asymmetrically punishes: if true=\$10 and predicted=\$20, MAPE=100%. But if true=\$20 and predicted=\$10, MAPE=50%. Same absolute error, different penalty.

**Alternatives when y=0 is possible:**
- **sMAPE (Symmetric MAPE):** uses average of |y| and |ŷ| in denominator
- **RMSPE:** root mean squared percentage error
- **MASE:** mean absolute scaled error (compares to a naive forecast)

In [ ]:
# ---------------------------------------------------------------
# MAPE asymmetry and zero-value problem demonstration
# ---------------------------------------------------------------
print("=== MAPE Asymmetry Demo ===")
cases = [
    ('True=$10, Pred=$20 (over-predict)',  10,  20),
    ('True=$20, Pred=$10 (under-predict)', 20,  10),
    ('True=$100, Pred=$90',               100,  90),
    ('True=$100, Pred=$110',              100, 110),
    ('True=$0.01, Pred=$10 (tiny true)',  0.01, 10),   # extreme MAPE for small values
]

for label, true_val, pred_val in cases:
    abs_err  = abs(true_val - pred_val)
    pct_err  = abs_err / abs(true_val) * 100
    print(f"  {label:42s} | Abs Error: ${abs_err:7.2f} | MAPE: {pct_err:8.1f}%")

print()
print("=== MAPE undefined when true = 0 ===")
y_with_zero = np.array([100.0, 50.0, 0.0, 75.0])   # one zero transaction
y_pred_zero = np.array([ 90.0, 60.0, 5.0, 70.0])
# Without epsilon guard, this would be division by zero
with np.errstate(divide='ignore', invalid='ignore'):
    raw_mape = np.abs((y_with_zero - y_pred_zero) / y_with_zero) * 100
print(f"  Per-sample MAPE values: {raw_mape}")  # inf for the zero transaction
print(f"  MAPE with epsilon guard: {mape_scratch(y_with_zero, y_pred_zero):.1f}%")

# sMAPE as a more symmetric alternative
def smape_scratch(y_true, y_pred, epsilon=1e-8):
    """Symmetric MAPE — denominates by average of |y| and |ŷ|."""
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2 + epsilon
    return np.mean(np.abs(y_true - y_pred) / denom) * 100

print(f"\n  Linear model MAPE:  {mape_scratch(y_test, y_pred_lr):.2f}%")
print(f"  Linear model sMAPE: {smape_scratch(y_test, y_pred_lr):.2f}%")

## Section 7: Residual Plots

Metrics give you single numbers. Residual plots tell you *where* and *how* your model is failing.

| Plot | What to Look For |
|------|------------------|
| **Actual vs Predicted** | Points along the diagonal y=x line |
| **Residuals vs Predicted** | Random horizontal cloud — no pattern or fan shape |
| **Residuals Histogram** | Roughly bell-shaped centered at 0 |

**Red flags:**
- Curved pattern → model is missing non-linearity
- Fan shape (wider spread at high values) → heteroscedasticity
- Heavy tails in histogram → outlier-dominated errors

In [ ]:
# ---------------------------------------------------------------
# Residual diagnostic plots — Linear vs Polynomial
# ---------------------------------------------------------------
residuals_lr   = y_test - y_pred_lr
residuals_poly = y_test - y_pred_poly

fig = plt.figure(figsize=(15, 9))
gs  = gridspec.GridSpec(2, 3, hspace=0.42, wspace=0.35)

model_data = [
    ('Linear Regression',     y_pred_lr,   residuals_lr,   'steelblue'),
    ('Polynomial (degree=2)', y_pred_poly, residuals_poly, 'mediumseagreen'),
]

for row, (name, y_pred, resid, color) in enumerate(model_data):
    # Panel 1: Actual vs Predicted
    ax1 = fig.add_subplot(gs[row, 0])
    ax1.scatter(y_test, y_pred, alpha=0.35, s=18, color=color)
    lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
    ax1.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect prediction')
    ax1.set_xlabel('Actual Amount ($)')
    ax1.set_ylabel('Predicted Amount ($)')
    ax1.set_title(f'{name}\nActual vs Predicted', fontweight='bold')
    ax1.legend(fontsize=9)

    # Panel 2: Residuals vs Predicted
    ax2 = fig.add_subplot(gs[row, 1])
    ax2.scatter(y_pred, resid, alpha=0.35, s=18, color=color)
    ax2.axhline(0, color='red', linestyle='--', linewidth=1.5)
    ax2.set_xlabel('Predicted Amount ($)')
    ax2.set_ylabel('Residual ($)')
    ax2.set_title(f'{name}\nResiduals vs Predicted', fontweight='bold')

    # Panel 3: Residual histogram
    ax3 = fig.add_subplot(gs[row, 2])
    ax3.hist(resid, bins=35, color=color, edgecolor='white', alpha=0.85)
    ax3.axvline(0, color='red', linestyle='--', linewidth=1.5)
    ax3.set_xlabel('Residual ($)')
    ax3.set_ylabel('Count')
    ax3.set_title(f'{name}\nResidual Distribution', fontweight='bold')

plt.suptitle('Residual Diagnostic Plots', fontsize=14, fontweight='bold')
plt.show()

## Section 8: Full Metric Comparison Table

We create four model variants (two models + two intentionally bad baselines) and compare all five metrics.

In [ ]:
# ---------------------------------------------------------------
# Build 4-model comparison table
# ---------------------------------------------------------------

# Baseline 1: always predict the mean of training set
y_pred_mean = np.full_like(y_test, fill_value=y_train.mean())

# Baseline 2: predict the median (robust baseline)
y_pred_median = np.full_like(y_test, fill_value=np.median(y_train))

all_models = {
    'Always-Mean Baseline':    (y_pred_mean,   1),
    'Always-Median Baseline':  (y_pred_median, 1),
    'Linear Regression':       (y_pred_lr,     3),
    'Polynomial (degree=2)':   (y_pred_poly,   9),
}

table_rows = []
for model_name, (y_pred, p) in all_models.items():
    table_rows.append({
        'Model':          model_name,
        'MAE ($)':        round(mae_scratch(y_test, y_pred), 2),
        'RMSE ($)':       round(rmse_scratch(y_test, y_pred), 2),
        'R²':             round(r2_scratch(y_test, y_pred), 4),
        'Adjusted R²':    round(adjusted_r2_scratch(y_test, y_pred, p), 4),
        'MAPE (%)':       round(mape_scratch(y_test, y_pred), 2),
    })

comparison_table = pd.DataFrame(table_rows).set_index('Model')
print("=== Full Metric Comparison Table ===")
print(comparison_table.to_string())
print()
print("Notes:")
print("  - Baselines have R²≈0 and negative adjusted R² (dividing by n-p-1 with p=1)")
print("  - Polynomial model: better MAE and RMSE, but watch adjusted R² if p grows further")
print("  - MAPE is scale-free — useful for comparing across different transaction scales")

In [ ]:
# ---------------------------------------------------------------
# Visual metric comparison: bar chart for MAE and RMSE,
# and R² / Adjusted R² comparison
# ---------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

model_labels = list(all_models.keys())
mae_vals  = comparison_table['MAE ($)'].values
rmse_vals = comparison_table['RMSE ($)'].values
r2_vals   = comparison_table['R²'].values
ar2_vals  = comparison_table['Adjusted R²'].values
mape_vals = comparison_table['MAPE (%)'].values

x = np.arange(len(model_labels))
width = 0.35

# MAE vs RMSE
axes[0].bar(x - width/2, mae_vals,  width, label='MAE',  color='steelblue',      alpha=0.85)
axes[0].bar(x + width/2, rmse_vals, width, label='RMSE', color='tomato',         alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(model_labels, rotation=20, ha='right', fontsize=9)
axes[0].set_ylabel('Error ($)')
axes[0].set_title('MAE vs RMSE by Model', fontweight='bold')
axes[0].legend()

# R² vs Adjusted R²
axes[1].bar(x - width/2, r2_vals,  width, label='R²',          color='mediumseagreen', alpha=0.85)
axes[1].bar(x + width/2, ar2_vals, width, label='Adjusted R²', color='darkorchid',     alpha=0.85)
axes[1].set_xticks(x)
axes[1].set_xticklabels(model_labels, rotation=20, ha='right', fontsize=9)
axes[1].set_ylabel('Score')
axes[1].set_title('R² vs Adjusted R² by Model', fontweight='bold')
axes[1].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].legend()

# MAPE
axes[2].bar(model_labels, mape_vals, color='goldenrod', alpha=0.85, edgecolor='white')
axes[2].set_xticklabels(model_labels, rotation=20, ha='right', fontsize=9)
axes[2].set_ylabel('MAPE (%)')
axes[2].set_title('MAPE by Model (% Error)', fontweight='bold')

plt.suptitle('Regression Metrics: 4-Model Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Section 9: Summary — When to Use Each Metric

| Metric | Best When | Avoid When |
|--------|-----------|------------|
| **MAE** | Interpretability matters; outliers shouldn't dominate | You need to heavily penalise large errors |
| **RMSE** | Large errors are especially costly; you want sensitivity to outliers | Outlier-heavy data and you care about typical performance |
| **R²** | Reporting explained variance; single model evaluation | Comparing models with different numbers of features |
| **Adjusted R²** | Comparing models with different feature counts | n is very small relative to p |
| **MAPE** | Stakeholder communication; scale-free comparison | Any true value is 0 or near 0 |

**Fraud detection tip:** For transaction amount prediction, report *both* RMSE (to flag when large fraud amounts are badly mis-estimated) *and* MAE (for typical performance). If RMSE >> MAE, you have outlier predictions that need investigation.

## Self-Check Questions

Work through these before moving to the next notebook.

---

**Q1.** Your regression model for transaction amounts has RMSE = \$2,500 and MAE = \$800. What does the large gap between these two metrics tell you about the errors?

<details>
<summary>Answer</summary>

The large gap (RMSE = $2,500 vs MAE = $800) indicates that the model makes a few **very large errors** that are inflating the RMSE. Since RMSE squares each error before averaging, a single prediction off by $10,000 contributes $100,000,000 to the sum — massively disproportionate to its frequency. The MAE tells you the *typical* error is only $800, which is reasonable; but RMSE reveals there are **outlier mis-predictions** — likely large fraudulent transactions the model completely missed. Action: investigate the worst-predicted transactions separately; they may all belong to a specific merchant category or time of day.

</details>

---

**Q2.** R² = 0.92 for your 3-feature model. You add 7 more random (noise) features and R² rises to 0.95. Is this the better model? What metric should you check instead?

<details>
<summary>Answer</summary>

No — this is **not** a better model. R² mathematically cannot decrease when you add features, even random ones. The rise from 0.92 to 0.95 is almost certainly caused by the model accidentally fitting the noise in the training data, which will not generalise to new transactions. You should check **Adjusted R²**. With 10 features vs 3 features (and the same n), the penalty term (n-1)/(n-p-1) increases substantially. If the 7 new features are truly useless, Adjusted R² will *decrease* relative to the 3-feature model — correctly identifying the original model as better. Always report Adjusted R² when the feature count differs across models.

</details>

---

**Q3.** MAPE is undefined when the true value is 0. Give a practical example from fraud detection where true transaction amounts of 0 would appear, making MAPE problematic.

<details>
<summary>Answer</summary>

Several realistic scenarios:

1. **Pre-authorisation transactions:** Some card networks send $0.00 "test" charges to verify a card is active before charging the real amount. A fraud model might include these.
2. **Refunds recorded as $0 net:** Some systems record a full refund followed by a $0 net amount in the transaction ledger.
3. **Free trials / promotional transactions:** A new subscription service charges $0.00 for the first month — this is a real, legitimate transaction with a $0 amount.
4. **Voided transactions:** A transaction that was initiated but immediately voided may appear as $0 in some datasets.

In any of these cases, |y_i - ŷ_i| / |y_i| = (some number) / 0, which is undefined (infinity). Even a $1 prediction error produces an infinite MAPE. Practical solutions: use MAE or RMSE instead, use sMAPE (denominator averages |y| and |ŷ|), or filter out zero-amount transactions from MAPE calculation while reporting them separately.

</details>